# Overhead plots

The idea is to create a plot for the CPU and latency overhead when there's an Agent actually taking decisions and when there is no Agent.

The first approximation is: when there is no agent --> when the agent picks always the same value, in that case I am assuming the cost is basically 0. I am saying this because we measure the CPU consumption of the Aggregate, and if the D value does not change there is not really much the Aggregate is doing.

Of course it would be more "correct" to do experiments with an Aggregate that is not connected at all with the RL framework part, but in the interest of time we start with this.

Now, we need to go back to the actual log data and see a bit what we have... the data is here:  `data/10/linearroad-CCR/5/600`. From the data:
- we have all the D values
- we have 100 episodes for the D value
- even when D is 10, the episode runs for some 80 seconds

Hence:
- we could take the middle 60 seconds from each D and from each episode and log them
- we already have the python script that "cuts" individual episodes from the single log file, so we could add an optional parameter that asks if we want to dump the individual logs (default: no)
- then we do the same for the RL agent experiment and we have what we need to start creating the new plots

To avoid messing with the data we already have, I am creating a copy of the folder so we create the new data in the new folder. I'm copying it into `data/exp016.22.overhead/linearroad/baseline`

- `mkdir data/exp016.22.overhead`
- `mkdir data/exp016.22.overhead/linearroad`
- `mkdir data/exp016.22.overhead/linearroad/baseline`
- `cp -r data/10/linearroad-CCR/5/600/* data/exp016.22.overhead/linearroad/baseline/`

Then I am running:
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad/baseline True 0` but changing `reward_pattern` to `"Old` to see what happens

This seems to work
Now I'm adding --dumpdata to `plot_experiment_stats_exp.py` to dump data too (if the paramater is passed)

This is how you run it:
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad/baseline True 0 1 2 3 4 5 6 7 8 9 10`

Now doing the same for synthetic

And now trying with the agent one, again:
- copy the data
- Run the script (this time with the New not the Old parameter)
- `mkdir data/exp016.22.overhead/linearroad/agent`
- `cp -r data/exp16-5/WELAW/linear/* data/exp016.22.overhead/linearroad/agent/`
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/linearroad True agent`
- `mkdir data/exp016.22.overhead/synthetic/agent`
- `cp -r data/exp16-5/WELAW/synthetic/* data/exp016.22.overhead/synthetic/agent/`
- `./scripts/create_plots_for_exp.sh data/exp016.22.overhead/synthetic True agent`



# ALL THE REST IS OLD

## What rewards do we get? Are these the expected ones?

- `python plotting/actions_rewards.py data/exp16.20_shepherding`

- From SPE always 0
- From Agent high variability, as expeted. Cannot say if negative but that can be checked later on with the following graphs

## Are the durations across actions the expected ones?

- `python plotting/actions_times.py data/exp16.20_shepherding`

Usual ones, yes

## Termination
- How many terminate because of CPU?
- How many terminate because of latency?
- How many terminate because of CPU & Latency?

Copy paste the following in temp.sh and run as temp.sh data/exp16.20_shepherding

```
find $1 -type f -name "python_agent.log" -exec bash -c '
  for file; do
    echo "Processing $file"
    cpu_count=$(grep -c "High cpu observed" "$file")
    latency_count=$(grep -c "High latency observed" "$file")
    echo "High CPU observed: $cpu_count"
    echo "High Latency observed: $latency_count"
  done
' bash {} +
```


```
Processing data/exp16.20//WELOB/linear/python_agent.log
High CPU observed: 73
High Latency observed: 1
Processing data/exp16.20//ELOB/linear/python_agent.log
High CPU observed: 108
High Latency observed: 21
Processing data/exp16.20//LOB/linear/python_agent.log
High CPU observed: 115
High Latency observed: 29
Processing data/exp16.20//WELAW/linear/python_agent.log
High CPU observed: 120
High Latency observed: 4
```
It's mostly CPU, and the longer the inter-action time, the higher the number of early termination because of CPU. The threshold is 90.0

### Prepare data

Copy paste the following in temp.sh and run 

```
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False WELOB/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/WELOB
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False ELOB/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/ELOB
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False LOB/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/LOB
./scripts/create_plots_for_exp.sh data/exp16.20_shepherding False WELAW/linear
python plotting/create_summary_data.py data/exp16.20_shepherding/WELAW
sed -i '' 's/linear/welob_linear/g' data/exp16.20_shepherding/WELOB/baselines_data.csv
sed -i '' 's/linear/elob_linear/g' data/exp16.20_shepherding/ELOB/baselines_data.csv
sed -i '' 's/linear/lob_linear/g' data/exp16.20_shepherding/LOB/baselines_data.csv
sed -i '' 's/linear/welaw_linear/g' data/exp16.20_shepherding/WELAW/baselines_data.csv
find data/exp16.20_shepherding/ -type f -name "baselines_data.csv" -exec cat {} + > data/exp16.20_shepherding/merged.csv
python plotting/plot_probs_evolution.py data/exp16.20_shepherding
python plotting/plot_cumulative_reward.py data/exp16.20_shepherding/ 1001 3100
```

## Steps and rewards

In [1]:
from IPython.display import display, Markdown

# Define your image folder path
path_to_image = "../data/exp16.20_shepherding/"

# Create a Markdown string with the table and images
markdown_table = f"""
<table>
    <tr>
        <td><img src="{path_to_image}WELOB_linear.steps.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB_linear.steps.png" alt="Image 2" width="450"/></td>
        <td><img src="{path_to_image}LOB_linear.steps.png" alt="Image 3" width="450"/></td>
        # <td><img src="{path_to_image}WELAW_linear.steps.png" alt="Image 3" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}WELOB_linear.reward.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB_linear.reward.png" alt="Image 2" width="450"/></td>
        <td><img src="{path_to_image}LOB_linear.reward.png" alt="Image 3" width="450"/></td>
        # <td><img src="{path_to_image}WELAW_linear.reward.png" alt="Image 3" width="450"/></td>
    </tr>
</table>
"""

# Render the Markdown
display(Markdown(markdown_table))



<table>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELOB_linear.steps.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB_linear.steps.png" alt="Image 2" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB_linear.steps.png" alt="Image 3" width="450"/></td>
        # <td><img src="../data/exp16.20_shepherding/WELAW_linear.steps.png" alt="Image 3" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELOB_linear.reward.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB_linear.reward.png" alt="Image 2" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB_linear.reward.png" alt="Image 3" width="450"/></td>
        # <td><img src="../data/exp16.20_shepherding/WELAW_linear.reward.png" alt="Image 3" width="450"/></td>
    </tr>
</table>


Seems consistent with stuff seen before

In [2]:
from IPython.display import display, Markdown

# Define your image folder path
path_to_image = "../data/exp16.20_shepherding/"

# Create a Markdown string with the table and images
markdown_table = f"""
<table>
    <tr>
        <td><img src="{path_to_image}WELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}ELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}ELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}LOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}LOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}LOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="{path_to_image}WELAW/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELAW/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="{path_to_image}WELAW/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
</table>
"""

# Render the Markdown
display(Markdown(markdown_table))



<table>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/ELOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/ELOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/LOB/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/LOB/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
    <tr>
        <td><img src="../data/exp16.20_shepherding/WELAW/linear/probs_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELAW/linear/probs_sum_probs.png" alt="Image 1" width="450"/></td>
        <td><img src="../data/exp16.20_shepherding/WELAW/linear/probs_entropy.png" alt="Image 1" width="450"/></td>
    </tr>
</table>


## final graph

- modify manually the merged file (remove the extra baseline,... lines)
- `python plotting/paper_plot_baselines_vs_multiple_agents_icpe_single_column.py data/10/linearroad-CCR/5/600 data/10/linearroad-CCR/5/600/lr_rate.csv data/exp16.20_shepherding/merged.csv data/exp16.20_shepherding/lr_baseline_vs_multiple_agents.pdf data/exp16.20_shepherding/lr_baseline_vs_multiple_agents.png linearroad welaw_linear welob_linear,elob_linear,lob_linear,welaw_linear WEL-OB,EL-OB,L-OB,WEL-AW`

In [3]:
from IPython.display import display, Markdown

# Define your image folder path
path_to_image = "../data/exp16.20_shepherding/"

# Create a Markdown string with the table and images
markdown_table = f"""
<table>
    <tr>
        <td><img src="{path_to_image}lr_baseline_vs_multiple_agents.png" alt="Image 1" width="450"/></td>
    </tr>
</table>
"""

# Render the Markdown
display(Markdown(markdown_table))



<table>
    <tr>
        <td><img src="../data/exp16.20_shepherding/lr_baseline_vs_multiple_agents.png" alt="Image 1" width="450"/></td>
    </tr>
</table>


Modify the script so that we mark the areas in which the moving average is below the threshold (will be relevant for linear I think)